## text2grammar Experiments testing


## Task1: 

Single Sentence -> Choose Correct Grammar Among # Number of Grammars inputs




In [15]:
import pandas as pd 
import os
import pickle
from datetime import datetime, timezone
from openai import OpenAI  # pip install openai
import nltk
from nltk.tokenize import sent_tokenize
import os
from pprint import pprint
import pandas as pd 
from mt_reasoning.utils import prompts_util, clients_util 
from tqdm import tqdm
import importlib
from dotenv import load_dotenv
import random
import string

load_dotenv()

source_df = pd.read_json("data/extraction_pdf/datasets/df_samples.jsonl", lines=True)

## uv run vllm serve /home/snt/projects_lujun/base_models/gemma-2-2b-it --host 0.0.0.0 --port 1997 --max-model-len 2048 --max-num-seqs 2 --gpu-memory-utilization 0.7


In [ ]:
importlib.reload(prompts_util)
importlib.reload(clients_util)

nltk.download('punkt')

## Open AI Settings
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
TEMPERATURE = float(os.environ.get("OPENAI_TEMPERATURE", "0.5"))


## VllM settings
model_vllm = os.environ.get("MODEL_VLLM", "/home/snt/projects_lujun/base_models/gemma-2-2b-it")
IP = os.environ.get("VLLM_IP", "0.0.0.0")
PORT = os.environ.get("VLLM_PORT", "1997")
server_url = f"http://{IP}:{PORT}/v1"
print (server_url)
vllm_client = OpenAI(base_url=server_url)

## Experimental Settings
grammar_list_size = 5
letters = list(string.ascii_uppercase)  # ['A', 'B', 'C', ..., 'Z']

time_now = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
output_dir = "/home/snt/projects_lujun/mt_reasoning/data/extraction_pdf/datasets"
output_path = os.path.join(output_dir, f"task1_{time_now}_{grammar_list_size}_{model_vllm.split('/')[-1]}.jsonl")

http://0.0.0.0:1997/v1


[nltk_data] Downloading package punkt to /home/snt/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [9]:
for index, row in tqdm(source_df.iterrows(), total=len(source_df)):
    grammar_desc = row['grammar_points_descriptions']
    opposite_source_grammar_list = source_df[source_df['grammar_points_descriptions'] != grammar_desc]['grammar_points_descriptions'].drop_duplicates().sample(grammar_list_size-1, random_state=42).tolist()
    full_list = [grammar_desc] + opposite_source_grammar_list
    random.shuffle(full_list)
    grammar_index = full_list.index(grammar_desc)

    assert grammar_index != -1, "Grammar description not found in the list."
    assert len(full_list) == grammar_list_size, "Grammar list size mismatch."

    option_labels = letters[:grammar_list_size]
    correct_grammar_letter = option_labels[grammar_index]
    labeled_grammar_list = [
        f"{label}. {desc}" for label, desc in zip(option_labels, full_list)
    ]
    input_dict = {
        "LUXEMBOURGISH_SENTENCE": row['luxembourg'],
        "ENGLISH_SENTENCE": row['english'],
        "LIST_GRAMMAR_DESCRIPTION": "\n".join(labeled_grammar_list),
    }


    output_dict, input_prompt = clients_util.generate_with_calling_api(
        client=vllm_client,
        system_prompt_template_path="prompts/system/system_prompt_translation.jinja",
        input_prompt_template_path="prompts/evaluation/prompt_grammar_classification_task_1.jinja",  # Use simple, complecated one confuse the models
        input_text_dict=input_dict,
        model=model_vllm,
    )
    
    row["input_prompt"] = input_prompt
    row["task1_dict"] = output_dict
    row["correct_grammar_letter"] = correct_grammar_letter
    updated_row = pd.DataFrame([row])
    if not os.path.exists(output_dir):  
        os.makedirs(output_dir)
    if not os.path.exists(output_path):
        updated_row.to_json(output_path, orient="records", lines=True)
    else:
        updated_row.to_json(output_path, orient="records", lines=True, mode="a")
    
    # print(output_dict)
    # print("----------------------------------------------")
    # pprint(output_dict, indent=2, width=150, sort_dicts=False)

100%|██████████| 2040/2040 [07:15<00:00,  4.68it/s]


In [16]:
result_df = pd.read_json("data/extraction_pdf/datasets/task1_20251002_122701_5.jsonl", lines=True)

In [17]:
num_correct = 0
total = len(result_df)
for index, row in result_df.iterrows():
    correct_grammar_letter = row['correct_grammar_letter']
    detected_grammar_letter = row['task1_dict'].get('grammar_selected', '').strip().upper()
    if correct_grammar_letter == detected_grammar_letter:
        num_correct += 1
print(f"Accuracy: {num_correct}/{total} = {num_correct/total:.2%}")

Accuracy: 1120/2040 = 54.90%


In [10]:
back_testing_df = pd.read_json("data/extraction_pdf/datasets/back_checking_20250930_170933.jsonl", lines=True)

In [14]:
translation_scores = []
is_contained_label_num = 0
total = len(back_testing_df)
for index, row in back_testing_df.iterrows():
    back_checking_dict = row['back_checking_dict']
    translation_score = back_checking_dict.get('translation_score', None)
    translation_scores.append(translation_score)
    is_contained_label = back_checking_dict.get('grammar_reflection', None)
    if is_contained_label == "Yes":
        is_contained_label_num += 1


print(f"Average translation score: {sum(translation_scores)/len(translation_scores):.4f}")
print(f"Number of sentences containing the grammar point: {is_contained_label_num}/{total} = {is_contained_label_num/total:.2%}")


Average translation score: 7.9426
Number of sentences containing the grammar point: 1917/2040 = 93.97%


In [13]:
back_checking_dict

{'translation_score': 9,
 'grammar_reflection': 'Yes',
 'false_sentence': 'Fuerscher hunn Quartieren gemellt, an deem Gemeinschaftszenteren, Schoulen a Geschäfter verschidde dominéierend Sproochen benotzen, wat variéiert Méiglechkeeten fir allerdags multilingual Kontakter ënner Awunner schaaft.',
 'false_reason': "Wrong relative connector agreement: used singular/neuter 'an deem' instead of the required dative plural 'an deenen' to match 'Quartieren'."}